In [1]:
import numpy as np, pandas as pd
from scipy.stats import beta

ca = pd.read_pickle('/home/ibm-user/shared/clean/ca3.pkl')
te = pd.read_pickle('/home/ibm-user/shared/clean/te3.pkl')
df = pd.concat([ca, te]).sort_values('ts').reset_index(drop=True)
df['win6'] = df['ts'].dt.floor('6h')

aw = (df.groupby(['win6','src_acct'])
        .agg(score=('score','max'), guilty=('laundering','max'))
        .reset_index())

test_wins = sorted(te['ts'].dt.floor('6h').unique())
all_wins = sorted(aw['win6'].unique())

def cp_upper(k, n, a=0.05):
    return 1.0 if k >= n else beta.ppf(1-a, k+1, n-k)

def pick_thr(inn_scores, cap):
    qs = np.quantile(inn_scores, np.linspace(0.9, 1.0, 2000))
    n = len(inn_scores)
    for t in np.unique(qs):
        k = (inn_scores > t).sum()
        if cp_upper(k, n) * 10000 <= cap:
            return t
    return qs[-1]

rows = []
for cap in [2, 5, 10]:
    for w in test_wins:
        i = all_wins.index(w)
        prev = all_wins[max(0, i-5):i-1]        # 4 windows, skipping the immediately preceding one
        hist = aw[aw.win6.isin(prev) & (aw.guilty == 0)]
        thr = pick_thr(hist.score.values, cap)
        cur = aw[aw.win6 == w]
        held = cur.score > thr
        harm = (held & (cur.guilty == 0)).sum() / len(cur) * 10000
        rec = (held & (cur.guilty == 1)).sum() / max((cur.guilty == 1).sum(), 1)
        rows.append((cap, w, len(prev), thr, harm, rec, (cur.guilty == 1).sum()))

r = pd.DataFrame(rows, columns=['cap','win','n_prev','thr','harm','recall','n_guilty'])
lines = []
for cap, g in r.groupby('cap'):
    lines.append(f"cap {cap}: windows={len(g)} mean_harm={g.harm.mean():.1f} worst={g.harm.max():.1f} "
                 f"over_cap={(g.harm > cap).sum()} of {len(g)} recall(pooled)={(g.recall*g.n_guilty).sum()/g.n_guilty.sum():.3f} "
                 f"min_prev_windows={g.n_prev.min()}")
txt = "\n".join(lines)
print(r.to_string()); print(); print(txt)
open('/home/ibm-user/shared/results/recal.txt','w').write(txt + "\n")

    cap                 win  n_prev       thr        harm    recall  n_guilty
0     2 2022-09-09 00:00:00       4  1.000000    2.090629  0.164557        79
1     2 2022-09-09 06:00:00       4  1.000000    0.000000  0.000000       104
2     2 2022-09-09 12:00:00       4  1.000000    0.000000  0.000000       144
3     2 2022-09-09 18:00:00       4  1.000000    0.000000  0.000000       103
4     2 2022-09-10 00:00:00       4  1.000000    0.000000  0.000000        78
5     2 2022-09-10 06:00:00       4  1.000000    0.489524  0.307692       117
6     2 2022-09-10 12:00:00       4  1.000000    1.553197  0.350746       134
7     2 2022-09-10 18:00:00       4  1.000000    0.000000  0.232558        86
8     5 2022-09-09 00:00:00       4  0.984499    3.658600  0.177215        79
9     5 2022-09-09 06:00:00       4  0.989196  128.838340  0.298077       104
10    5 2022-09-09 12:00:00       4  0.996287    6.401853  0.437500       144
11    5 2022-09-09 18:00:00       4  0.999652    3.202391  0.300

299